In [1]:
import strahler
import os
os.environ["STRAHLER_PM_BACKEND"] = "simd"
from importlib.metadata import version as package_version
if package_version("strahler") != strahler.version():
    raise RuntimeError("Package/library version mismatch. Reinstall strahler and restart the kernel.")
print("strahler package and library versions match")

from IPython.display import HTML, display
from notebook_support import HERE, table
from lexicographic_nester import BACKENDS, host_svg
from family_benchmarks import (ORDERS, recorded, result_text, fixed_maps, shape_html)


strahler package and library versions match


# Ordinary ladders: increase the rung count

We use the original PGSolver **laddergame(n)** family collected in
[Keiren's CC0 parity-game dataset](https://doi.org/10.6084/m9.figshare.6004130.v1).
[Keiren's survey](https://www.jeroenkeiren.nl/assets/publications/Kei2015.pdf)
attributes the PGSolver family to Friedmann. The archive supplies nine increasing
parameters: **100, 200, 500, 1,000, 2,000, 5,000, 10,000, 20,000 and 50,000**.
These are the ordinary `laddergame` files, with the exact archive members and
checksums bundled alongside the notebook.

Each instance has **2n vertices and 4n edges**, with priorities 0 and 1.
Increasing n grows the graph while its height bounds stay fixed. AD's heights
are 2/1; STRPM's are 2/2. This makes the family useful for separating graph
scaling from a need for taller host trees.

In [2]:
info, recording = recorded("ladder")
games = info["games"]
table(["case", "parameter", "vertices", "edges", "AD h E/O", "STRPM h E/O", "AD k cap", "STRPM k cap", "t cap"],
      [[g["label"], g["parameter"], g["vertices"], g["edges"], g["AD_height"],
        [d + 1 for d in g["PM_depth"]], g["AD_k_max"], g["PM_k_max"], g["t_max"]]
       for g in games])

case,parameter,vertices,edges,AD h E/O,STRPM h E/O,AD k cap,STRPM k cap,t cap
ladder n=100,100,200,400,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",7
ladder n=200,200,400,800,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",8
ladder n=500,500,1000,2000,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",9
ladder n=1000,1000,2000,4000,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",10
ladder n=2000,2000,4000,8000,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",11
ladder n=5000,5000,10000,20000,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",13
ladder n=10000,10000,20000,40000,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",14
ladder n=20000,20000,40000,80000,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",15
ladder n=50000,50000,100000,200000,"[2, 1]","[2, 2]","[2, 1]","[2, 2]",16


## Start with the smallest host

The unary pair **(k,t)=(1,0)** gives a chain. These are its only two host heights
in this experiment. A large vertex count does not itself force a large branching
host: the fixed runs below test what is actually needed.

In [3]:
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:16px">' +
             host_svg(1, 0, 1, "AD Odd: h=1") +
             host_svg(1, 0, 2, "AD Even and STRPM E/O: h=2") + '</div>'))

## A winning decomposition

Zielonka's answer for the smallest archived game. Node labels count the
vertices in H, T and S; the drawing keeps just the tree shape.


In [4]:
first_reference = next(r for r in recording["references"] if r["game"] == games[0]["id"])
table(["reference case", "outcome", "seconds", "stage", "won E/O"],
      [[r["game"], r["outcome"], f"{r['seconds']:.2f}", r["stage"], r.get("won", "unknown")]
       for r in recording["references"]])
display(HTML(shape_html(first_reference.get("reference_shape"))))

reference case,outcome,seconds,stage,won E/O
ladder-100,solved,0.10,JSON reduction and winner validation,"[100, 100]"
ladder-200,solved,0.10,JSON reduction and winner validation,"[200, 200]"
ladder-500,solved,0.10,JSON reduction and winner validation,"[500, 500]"
ladder-1000,solved,0.11,JSON reduction and winner validation,"[1000, 1000]"
ladder-2000,solved,0.11,JSON reduction and winner validation,"[2000, 2000]"
ladder-5000,solved,0.14,JSON reduction and winner validation,"[5000, 5000]"
ladder-10000,solved,0.17,JSON reduction and winner validation,"[10000, 10000]"
ladder-20000,solved,0.24,JSON reduction and winner validation,"[20000, 20000]"
ladder-50000,solved,0.44,JSON reduction and winner validation,"[50000, 50000]"


## Three search orders: one minute per case

We compare AD, symmetric AD and Strahler PM, each with and without iterative
shaving. Preprocessing and same-parity compression are off.

Lexicographic order exhausts t before increasing k. Ratio favours balanced
pairs and starts at (1,1). Table order uses the
[precomputed host sizes](../HOST-ORDER.md), then falls back to ratio.

Runs have **60 seconds and 2 GiB**, with at most two solvers together. Times
include input, solving, output and winner checks against Zielonka. Cells show
the final pair and seconds; +iter also includes wins from earlier rounds.


In [5]:
by_case = {(r["game"], r["backend"], r["order"]): r for r in recording["rows"]}
for order in ORDERS:
    print(order)
    table(["case", "vertices", *BACKENDS],
          [[g["label"], g["vertices"],
            *[result_text(by_case[g["id"], b, order]) for b in BACKENDS]] for g in games])

lexicographic


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
ladder n=100,200,"(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.10 s","(1, 0), 0.10 s"
ladder n=200,400,"(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.10 s"
ladder n=500,1000,"(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.10 s"
ladder n=1000,2000,"(1, 0), 0.12 s","(1, 0), 0.12 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s"
ladder n=2000,4000,"(1, 0), 0.12 s","(1, 0), 0.12 s","(1, 0), 0.12 s","(1, 0), 0.12 s","(1, 0), 0.11 s","(1, 0), 0.11 s"
ladder n=5000,10000,"(1, 0), 0.14 s","(1, 0), 0.15 s","(1, 0), 0.14 s","(1, 0), 0.14 s","(1, 0), 0.13 s","(1, 0), 0.13 s"
ladder n=10000,20000,"(1, 0), 0.19 s","(1, 0), 0.18 s","(1, 0), 0.18 s","(1, 0), 0.18 s","(1, 0), 0.16 s","(1, 0), 0.15 s"
ladder n=20000,40000,"(1, 0), 0.26 s","(1, 0), 0.27 s","(1, 0), 0.27 s","(1, 0), 0.27 s","(1, 0), 0.21 s","(1, 0), 0.21 s"
ladder n=50000,100000,"(1, 0), 0.49 s","(1, 0), 0.51 s","(1, 0), 0.50 s","(1, 0), 0.51 s","(1, 0), 0.36 s","(1, 0), 0.36 s"


ratio


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
ladder n=100,200,"(1, 1), 0.10 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.10 s","(1, 1), 0.10 s","(1, 1), 0.10 s"
ladder n=200,400,"(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.10 s","(1, 1), 0.10 s"
ladder n=500,1000,"(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s"
ladder n=1000,2000,"(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s","(1, 1), 0.11 s"
ladder n=2000,4000,"(1, 1), 0.12 s","(1, 1), 0.12 s","(1, 1), 0.12 s","(1, 1), 0.12 s","(1, 1), 0.12 s","(1, 1), 0.12 s"
ladder n=5000,10000,"(1, 1), 0.14 s","(1, 1), 0.14 s","(1, 1), 0.14 s","(1, 1), 0.14 s","(1, 1), 0.13 s","(1, 1), 0.13 s"
ladder n=10000,20000,"(1, 1), 0.18 s","(1, 1), 0.18 s","(1, 1), 0.18 s","(1, 1), 0.18 s","(1, 1), 0.16 s","(1, 1), 0.16 s"
ladder n=20000,40000,"(1, 1), 0.26 s","(1, 1), 0.26 s","(1, 1), 0.26 s","(1, 1), 0.27 s","(1, 1), 0.21 s","(1, 1), 0.21 s"
ladder n=50000,100000,"(1, 1), 0.49 s","(1, 1), 0.50 s","(1, 1), 0.50 s","(1, 1), 0.51 s","(1, 1), 0.37 s","(1, 1), 0.36 s"


table


case,vertices,AD,AD+iter,AD-sym,AD-sym+iter,PM,PM+iter
ladder n=100,200,"(1, 0), 0.11 s","(1, 0), 0.10 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.10 s","(1, 0), 0.11 s"
ladder n=200,400,"(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.10 s","(1, 0), 0.11 s"
ladder n=500,1000,"(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s"
ladder n=1000,2000,"(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s","(1, 0), 0.11 s"
ladder n=2000,4000,"(1, 0), 0.12 s","(1, 0), 0.12 s","(1, 0), 0.12 s","(1, 0), 0.12 s","(1, 0), 0.11 s","(1, 0), 0.11 s"
ladder n=5000,10000,"(1, 0), 0.14 s","(1, 0), 0.14 s","(1, 0), 0.14 s","(1, 0), 0.14 s","(1, 0), 0.13 s","(1, 0), 0.13 s"
ladder n=10000,20000,"(1, 0), 0.18 s","(1, 0), 0.18 s","(1, 0), 0.18 s","(1, 0), 0.19 s","(1, 0), 0.15 s","(1, 0), 0.15 s"
ladder n=20000,40000,"(1, 0), 0.26 s","(1, 0), 0.26 s","(1, 0), 0.26 s","(1, 0), 0.27 s","(1, 0), 0.21 s","(1, 0), 0.20 s"
ladder n=50000,100000,"(1, 0), 0.49 s","(1, 0), 0.50 s","(1, 0), 0.50 s","(1, 0), 0.51 s","(1, 0), 0.36 s","(1, 0), 0.36 s"


## Fixed pairs at the published endpoints

We map **n=100 and n=50,000**, k=1…2 and t=0…6, for the same four fixed
methods as notebook 08. Each run has 60 seconds and 2 GiB. This checks both
ends of the scaling series without repeating a dense grid at every rung count.

If (1,0) solves in a fresh fixed run, it is already the unique componentwise
minimum over the whole admissible grid. The remaining cells can be skipped for
finding minima, while their individual solver outcomes stay unmeasured.
Otherwise the collector continues checking the declared rectangle, including
predecessors of any later solve. Stars mark confirmed minima; diamonds have
open predecessors; amber resource limits leave the answer unknown.

Success is **not monotone in (k,t)**. We cap the scan by skipping larger
pairs after a solve. At fixed height, a solve at (k,t) transfers to (K,T) if
**K=k and T≥t**, or **K>k and T≥t+k−1**. A solved chain (k=1 or t=0)
transfers to every host. See the [embedding proof](../docs/host-monotonicity.md).

Green **⇒** cells follow from this bound and have no measured time. Other green symbols are measured solves. Gray cells remain unknown; amber cells
retain their measured limits.


In [6]:
maps, minima = fixed_maps(info, recording)
display(HTML(maps))
table(["case", "method", "confirmed minimal (k,t)", "observed nondominated", "time/memory limits"], minima)

case,method,"confirmed minimal (k,t)",observed nondominated,time/memory limits
ladder n=100,AD-sym,"[(1, 0)]","[(1, 0)]",[]
ladder n=100,AD-sym+iter,"[(1, 0)]","[(1, 0)]",[]
ladder n=100,PM,"[(1, 0)]","[(1, 0)]",[]
ladder n=100,PM+iter,"[(1, 0)]","[(1, 0)]",[]
ladder n=50000,AD-sym,"[(1, 0)]","[(1, 0)]",[]
ladder n=50000,AD-sym+iter,"[(1, 0)]","[(1, 0)]",[]
ladder n=50000,PM,"[(1, 0)]","[(1, 0)]",[]
ladder n=50000,PM+iter,"[(1, 0)]","[(1, 0)]",[]


All **162 adaptive cases** solve within a minute. At both published
endpoints, every fixed method solves at **(1,0)**. Growing from 200 to
100,000 vertices does not require extra branching in these runs.

The measured PM hosts are unary, so they use the scalar fallback despite
requesting SIMD. Ratio starts at (1,1), which is also a chain when k=1;
lexicographic and table order start at (1,0).

Sources and commands for rerunning the experiment are in
[ladder/README.md](ladder/README.md).


## A small live check

Solve the smallest archived ladder with table search and compare its PM
winners against Zielonka.


In [7]:
import bz2
from fixed_pair_maps import winning_sets
small = games[0]
with strahler.Game.from_text(bz2.decompress((HERE / "ladder" / small["file"]).read_bytes()).decode()) as game:
    expected = winning_sets(strahler.decompose(game, "zielonka").data, "AD")
    pm = strahler.progress_measures(game, algorithm="strahler", search="table", iter_shave=True)
    assert pm["rounds"][-1]["complete"]
    assert winning_sets(pm, "PM") == expected
    print("Smallest ladder: STRPM table search agrees with Zielonka.")

Smallest ladder: STRPM table search agrees with Zielonka.
